# TP 2 · Couche Silver avec Spark et Apache Iceberg

## Étape 2 · Créer la table Silver et écrire les données nettoyées

In [1]:
import time
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName("tp2").getOrCreate()

# Utilitaire : relève fichiers / snapshots / lignes après chaque étape
results = []
def stats(label):
    nb_f = spark.sql("SELECT count(*) AS c FROM demo.silver.orders.files").collect()[0].c
    nb_s = spark.sql("SELECT count(*) AS c FROM demo.silver.orders.snapshots").collect()[0].c
    nb_l = spark.table("demo.silver.orders").count()
    results.append((label, nb_f, nb_s, nb_l))
    print(f"[{label}] fichiers: {nb_f} | snapshots: {nb_s} | lignes: {nb_l}")

26/10/18 17:16:05 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


### Départ propre
Supprime la table si elle existe déjà, pour que le notebook puisse être relancé en entier sans doubler les données.

In [2]:
try:
    spark.sql("DROP TABLE IF EXISTS demo.silver.orders PURGE")
    print("ancienne table supprimée (si elle existait)")
except Exception as e:
    print("rien à supprimer :", str(e)[:120])

ancienne table supprimée (si elle existait)


### Cellule 1 · Espace de noms et table

In [3]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.silver")
spark.sql("""
CREATE TABLE IF NOT EXISTS demo.silver.orders (
  order_id STRING, customer_id BIGINT, order_ts TIMESTAMP,
  city STRING, amount DECIMAL(10,2), status STRING,
  _ingested_at TIMESTAMP)
USING iceberg
PARTITIONED BY (days(order_ts))
TBLPROPERTIES ('write.parquet.compression-codec' = 'zstd')
""")

DataFrame[]

### Cellule 2 · Nettoyer et écrire

In [4]:
raw = spark.read.parquet("/home/iceberg/notebooks/data/orders/")
silver = (raw
    .dropDuplicates(["order_id"])                       # unicité
    .filter(F.col("amount") > 0)                        # validité
    .withColumn("order_ts", F.col("order_ts").cast("timestamp"))
    .withColumn("amount", F.col("amount").cast("decimal(10,2)"))
    .withColumn("city", F.initcap(F.trim("city")))      # normalisation
    .withColumn("status", F.upper("status"))
    .drop("_batch_id"))
# Garde-fou : cette cellule ne doit être exécutée qu'une seule fois (sinon les lignes seraient doublées)
if spark.table("demo.silver.orders").count() > 0:
    raise Exception("La table contient déjà des données. Faire DROP TABLE demo.silver.orders PURGE, relancer la cellule 1, puis cette cellule.")
silver.writeTo("demo.silver.orders").append()
spark.table("demo.silver.orders").groupBy("status").count().show()
spark.sql("SELECT count(*) AS lignes, count(DISTINCT order_id) AS ids_distincts FROM demo.silver.orders").show()

+---------+-----+
|   status|count|
+---------+-----+
|     PAID|66772|
|CANCELLED|66573|
|  SHIPPED|66655|
+---------+-----+



[Stage 23:===================================================>    (21 + 2) / 23]

+------+-------------+
|lignes|ids_distincts|
+------+-------------+
|200000|       200000|
+------+-------------+



### Cellule 3 · Tables de métadonnées

In [5]:
spark.sql("SELECT * FROM demo.silver.orders.snapshots").show(truncate=False)
spark.sql("""SELECT partition, record_count, file_size_in_bytes
             FROM demo.silver.orders.files LIMIT 10""").show(truncate=False)
spark.sql("SELECT count(*) AS nb_fichiers FROM demo.silver.orders.files").show()
stats("Étape 2 : après écriture Silver")

+-----------------------+-------------------+---------+---------+----------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id|operation|manifest_list                                                                                             |summary                                                                                                                                                                           

### Questions 1 à 3

In [6]:
# Q1 : nombre de partitions (jours) distinctes
spark.sql("SELECT count(*) AS nb_partitions FROM demo.silver.orders.partitions").show()

+-------------+
|nb_partitions|
+-------------+
|           91|
+-------------+



In [7]:
# Q2 : contenu du metadata.json courant (via le catalogue REST)
import requests, json
try:
    r = requests.get("http://rest:8181/v1/namespaces/silver/tables/orders").json()
    m = r["metadata"]
    print("fichier metadata :", r["metadata-location"])
    print("snapshot courant :", m["current-snapshot-id"])
    print("schéma           :", [(f["id"], f["name"], f["type"]) for f in m["schemas"][-1]["fields"]])
    print("partitionnement  :", json.dumps(m["partition-specs"][-1]))
except Exception as e:
    print("Lecture via REST impossible :", e)
    print("-> télécharger le dernier *.metadata.json depuis la console MinIO (warehouse/silver/orders/metadata/)")

fichier metadata : s3://warehouse/silver/orders/metadata/00001-1853aef1-8830-415a-b9cd-8c6c8c41fc2b.metadata.json
snapshot courant : 1504196549823532520
schéma           : [(1, 'order_id', 'string'), (2, 'customer_id', 'long'), (3, 'order_ts', 'timestamptz'), (4, 'city', 'string'), (5, 'amount', 'decimal(10, 2)'), (6, 'status', 'string'), (7, '_ingested_at', 'timestamptz')]
partitionnement  : {"spec-id": 0, "fields": [{"name": "order_ts_day", "transform": "day", "source-id": 3, "field-id": 1000}]}


In [8]:
# Q3 : plan d'exécution avec filtre sur order_ts (colonne source)
plan = spark.sql("""EXPLAIN SELECT count(*) FROM demo.silver.orders
                    WHERE order_ts >= current_date() - INTERVAL 7 DAYS""").collect()[0][0]
print(plan)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[sum(agg_func_0#678L)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=764]
      +- HashAggregate(keys=[], functions=[partial_sum(agg_func_0#678L)])
         +- Project [count(*)#679L AS agg_func_0#678L]
            +- LocalTableScan [count(*)#679L]




In [9]:
# Q3 (variante) : plan avec filtre sur order_ts et une colonne lue (SUM) -> le BatchScan apparait avec le filtre pousse
# (avec count(*), Iceberg repond depuis les metadonnees : le plan se reduit a un LocalTableScan)
plan = spark.sql("""EXPLAIN SELECT sum(amount) FROM demo.silver.orders
                    WHERE order_ts >= current_date() - INTERVAL 7 DAYS""").collect()[0][0]
print(plan)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[sum(amount#691)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=785]
      +- HashAggregate(keys=[], functions=[partial_sum(amount#691)])
         +- Project [amount#691]
            +- BatchScan demo.silver.orders[order_ts#689, amount#691] demo.silver.orders (branch=null) [filters=order_ts IS NOT NULL, order_ts >= 1791676800000000, groupedBy=] RuntimeFilters: []




## Étape 3 · Upsert avec MERGE INTO

In [10]:
spark.sql("SELECT count(*) AS nb_fichiers FROM demo.silver.orders.files").show()
spark.sql("SELECT snapshot_id, committed_at, operation FROM demo.silver.orders.snapshots ORDER BY committed_at").show(truncate=False)

def make_updates():
    """1 000 commandes PAID -> SHIPPED + 500 nouvelles commandes (figé avec localCheckpoint)"""
    df = spark.sql("""
        SELECT * FROM (
          SELECT order_id, customer_id, order_ts, city, amount,
                 'SHIPPED' AS status, current_timestamp() AS _ingested_at
          FROM demo.silver.orders WHERE status = 'PAID' LIMIT 1000)
        UNION ALL
        SELECT * FROM (
          SELECT uuid() AS order_id, customer_id, current_timestamp() AS order_ts, city, amount,
                 'PAID' AS status, current_timestamp() AS _ingested_at
          FROM demo.silver.orders LIMIT 500)
    """).localCheckpoint()          # fige le lot : plan déterministe (uuid() ne change plus)
    df.createOrReplaceTempView("updates")
    return df

MERGE_SQL = """
MERGE INTO demo.silver.orders t
USING updates s
ON t.order_id = s.order_id
WHEN MATCHED AND t.status <> s.status THEN
  UPDATE SET t.status = s.status, t._ingested_at = s._ingested_at
WHEN NOT MATCHED THEN INSERT *
"""

+-----------+
|nb_fichiers|
+-----------+
|         91|
+-----------+

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|1504196549823532520|2026-10-18 17:18:09.047|append   |
+-------------------+-----------------------+---------+



### MERGE en copy-on-write (mode par défaut)

In [11]:
upd1 = make_updates()
t0 = time.time()
spark.sql(MERGE_SQL)
t_cow = time.time() - t0
print(f"MERGE copy-on-write : {t_cow:.1f} s")
stats("Étape 3 : après MERGE copy-on-write")
spark.sql("""SELECT snapshot_id, committed_at, operation,
                    summary['added-records'], summary['deleted-records'],
                    summary['added-data-files'], summary['deleted-data-files']
             FROM demo.silver.orders.snapshots ORDER BY committed_at""").show(truncate=False)

MERGE copy-on-write : 11.1 s
[Étape 3 : après MERGE copy-on-write] fichiers: 92 | snapshots: 2 | lignes: 200500
+-------------------+-----------------------+---------+----------------------+------------------------+-------------------------+---------------------------+
|snapshot_id        |committed_at           |operation|summary[added-records]|summary[deleted-records]|summary[added-data-files]|summary[deleted-data-files]|
+-------------------+-----------------------+---------+----------------------+------------------------+-------------------------+---------------------------+
|1504196549823532520|2026-10-18 17:18:09.047|append   |200000                |NULL                    |91                       |NULL                       |
|545040703085241320 |2026-10-18 17:19:07.822|overwrite|4789                  |4289                    |3                        |2                          |
+-------------------+-----------------------+---------+----------------------+--------------------

### Expérience : merge-on-read

In [12]:
spark.sql("""ALTER TABLE demo.silver.orders SET TBLPROPERTIES (
  'write.merge.mode' = 'merge-on-read',
  'write.update.mode' = 'merge-on-read',
  'write.delete.mode' = 'merge-on-read')""")
upd2 = make_updates()          # nouveau lot : 1 000 autres PAID + 500 nouvelles commandes
t0 = time.time()
spark.sql(MERGE_SQL)
t_mor = time.time() - t0
print(f"MERGE merge-on-read : {t_mor:.1f} s")
stats("Étape 3 : après MERGE merge-on-read")
spark.sql("""SELECT snapshot_id, operation,
                    summary['added-data-files'] AS added_data, summary['deleted-data-files'] AS deleted_data,
                    summary['added-delete-files'] AS added_delete_files,
                    summary['added-position-deletes'] AS pos_deletes
             FROM demo.silver.orders.snapshots ORDER BY committed_at""").show(truncate=False)
spark.sql("SELECT content, count(*) AS nb FROM demo.silver.orders.files GROUP BY content").show()

MERGE merge-on-read : 7.2 s


[Étape 3 : après MERGE merge-on-read] fichiers: 97 | snapshots: 3 | lignes: 201000
+-------------------+---------+----------+------------+------------------+-----------+
|snapshot_id        |operation|added_data|deleted_data|added_delete_files|pos_deletes|
+-------------------+---------+----------+------------+------------------+-----------+
|1504196549823532520|append   |91        |NULL        |NULL              |NULL       |
|545040703085241320 |overwrite|3         |2           |NULL              |NULL       |
|6050036303135822727|overwrite|3         |NULL        |2                 |1000       |
+-------------------+---------+----------+------------+------------------+-----------+

+-------+---+
|content| nb|
+-------+---+
|      1|  2|
|      0| 95|
+-------+---+



### Idempotence : rejouer le MERGE avec la même vue

In [13]:
before = spark.table("demo.silver.orders").count()
spark.sql(MERGE_SQL)          # même vue 'updates' (upd2, en cache)
after = spark.table("demo.silver.orders").count()
print("lignes avant :", before, "| après :", after)
stats("Étape 3 : après re-MERGE identique")
spark.sql("""SELECT snapshot_id, operation, summary['added-records'], summary['changed-partition-count']
             FROM demo.silver.orders.snapshots ORDER BY committed_at DESC LIMIT 1""").show(truncate=False)

lignes avant : 201000 | après : 201000


[Étape 3 : après re-MERGE identique] fichiers: 97 | snapshots: 4 | lignes: 201000
+-------------------+---------+----------------------+--------------------------------+
|snapshot_id        |operation|summary[added-records]|summary[changed-partition-count]|
+-------------------+---------+----------------------+--------------------------------+
|6119534653118430151|overwrite|NULL                  |0                               |
+-------------------+---------+----------------------+--------------------------------+



## Étape 4 · Time travel, rollback et évolution du schéma

In [14]:
snaps = spark.sql("SELECT snapshot_id, committed_at, operation FROM demo.silver.orders.snapshots ORDER BY committed_at").collect()
for s in snaps: print(s.snapshot_id, s.committed_at, s.operation)
snap_before_merge = snaps[0].snapshot_id          # l'append initial, avant tout MERGE
ts_before_merge = snaps[0].committed_at
print("snapshot avant MERGE :", snap_before_merge)

1504196549823532520 2026-10-18 17:18:09.047000 append
545040703085241320 2026-10-18 17:19:07.822000 overwrite
6050036303135822727 2026-10-18 17:19:21.485000 overwrite
6119534653118430151 2026-10-18 17:19:36.073000 overwrite
snapshot avant MERGE : 1504196549823532520


In [15]:
print("--- état AVANT le MERGE (VERSION AS OF) ---")
spark.sql(f"SELECT status, count(*) AS nb FROM demo.silver.orders VERSION AS OF {snap_before_merge} GROUP BY status ORDER BY status").show()
print("--- état ACTUEL ---")
spark.sql("SELECT status, count(*) AS nb FROM demo.silver.orders GROUP BY status ORDER BY status").show()
spark.sql(f"SELECT count(*) AS nb FROM demo.silver.orders TIMESTAMP AS OF '{ts_before_merge}'").show()

--- état AVANT le MERGE (VERSION AS OF) ---


+---------+-----+
|   status|   nb|
+---------+-----+
|CANCELLED|66573|
|     PAID|66772|
|  SHIPPED|66655|
+---------+-----+

--- état ACTUEL ---


+---------+-----+
|   status|   nb|
+---------+-----+
|CANCELLED|66573|
|     PAID|65772|
|  SHIPPED|68655|
+---------+-----+

+------+
|    nb|
+------+
|200000|
+------+



### Erreur simulée puis rollback

In [16]:
snap_before_delete = spark.sql("SELECT snapshot_id FROM demo.silver.orders.snapshots ORDER BY committed_at DESC LIMIT 1").collect()[0].snapshot_id
n_before = spark.table("demo.silver.orders").count()
spark.sql("DELETE FROM demo.silver.orders WHERE status = 'SHIPPED'")
n_deleted = spark.table("demo.silver.orders").count()
print("lignes avant DELETE :", n_before, "| après DELETE :", n_deleted)
spark.sql(f"CALL demo.system.rollback_to_snapshot('silver.orders', {snap_before_delete})").show()
print("lignes après rollback :", spark.table("demo.silver.orders").count())
stats("Étape 4 : après DELETE + rollback")

lignes avant DELETE : 201000 | après DELETE : 132345
+--------------------+-------------------+
|previous_snapshot_id|current_snapshot_id|
+--------------------+-------------------+
| 7311832242878411802|6119534653118430151|
+--------------------+-------------------+



lignes après rollback : 201000


[Stage 159:================================================>      (22 + 3) / 25]

[Étape 4 : après DELETE + rollback] fichiers: 97 | snapshots: 5 | lignes: 201000


### Évolution du schéma (sans réécriture)

In [17]:
spark.sql("ALTER TABLE demo.silver.orders ADD COLUMN channel STRING")
spark.sql("ALTER TABLE demo.silver.orders RENAME COLUMN city TO delivery_city")
spark.sql("ALTER TABLE demo.silver.orders ALTER COLUMN amount TYPE DECIMAL(12,2)")
spark.sql("DESCRIBE TABLE demo.silver.orders").show(truncate=False)
spark.sql("SELECT order_id, delivery_city, amount, channel FROM demo.silver.orders LIMIT 5").show()
stats("Étape 4 : après évolution du schéma")

+--------------+--------------+-------+
|col_name      |data_type     |comment|
+--------------+--------------+-------+
|order_id      |string        |NULL   |
|customer_id   |bigint        |NULL   |
|order_ts      |timestamp     |NULL   |
|delivery_city |string        |NULL   |
|amount        |decimal(12,2) |NULL   |
|status        |string        |NULL   |
|_ingested_at  |timestamp     |NULL   |
|channel       |string        |NULL   |
|              |              |       |
|# Partitioning|              |       |
|Part 0        |days(order_ts)|       |
+--------------+--------------+-------+

+--------------------+-----------------+------+-------+
|            order_id|    delivery_city|amount|channel|
+--------------------+-----------------+------+-------+
|107f5c85-da7a-4ab...|            David|207.36|   NULL|
|13e3cfce-9d32-4e1...|Foucher-les-bains|326.42|   NULL|
|15bab1b4-82e5-4c1...|    Saint Alixdan|554.23|   NULL|
|1bb56609-ceba-47d...|   Sainte Suzanne|894.31|   NULL|
|1cc9fa

[Stage 169:=====================================>                 (17 + 8) / 25]

[Étape 4 : après évolution du schéma] fichiers: 97 | snapshots: 5 | lignes: 201000


In [18]:
# Test : réduire DECIMAL(12,2) -> DECIMAL(10,2) doit être refusé
try:
    spark.sql("ALTER TABLE demo.silver.orders ALTER COLUMN amount TYPE DECIMAL(10,2)")
except Exception as e:
    print("Refusé comme prévu :", str(e)[:300])

Refusé comme prévu : [NOT_SUPPORTED_CHANGE_COLUMN] ALTER TABLE ALTER/CHANGE COLUMN is not supported for changing `demo`.`silver`.`orders`'s column `amount` with type "DECIMAL(12,2)" to `amount` with type "DECIMAL(10,2)".; line 1 pos 0;
AlterColumn resolvedfieldname(StructField(amount,DecimalType(12,2),true)), DecimalTyp


### Évolution du partitionnement : jour -> mois

In [19]:
spark.sql("ALTER TABLE demo.silver.orders REPLACE PARTITION FIELD order_ts_day WITH months(order_ts)")
spark.sql("SELECT * FROM demo.silver.orders.partitions").show(truncate=False)

+------------------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|partition         |spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at        |last_updated_snapshot_id|
+------------------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|{2026-10-02, NULL}|0      |2329        |1         |78346                        |0                           |0                         |0                           |0                         |2026-10-18 17:18:09.047|1504196549823532520     |
|{2026-09-07, NULL}|0   

In [20]:
# Petit lot de test pour voir où vont les nouvelles données
spark.sql("""INSERT INTO demo.silver.orders
             SELECT uuid(), 1, current_timestamp(), 'Test-Ville', CAST(10.00 AS DECIMAL(12,2)), 'PAID', current_timestamp(), 'web'""")
spark.sql("SELECT partition, record_count FROM demo.silver.orders.files ORDER BY partition DESC LIMIT 5").show(truncate=False)
stats("Étape 4 : après changement de partitionnement")

+------------------+------------+
|partition         |record_count|
+------------------+------------+
|{2026-10-18, NULL}|500         |
|{2026-10-18, NULL}|500         |
|{2026-10-12, NULL}|1662        |
|{2026-10-11, NULL}|2208        |
|{2026-10-10, NULL}|2243        |
+------------------+------------+



[Stage 183:==============================>                        (14 + 8) / 25]

[Étape 4 : après changement de partitionnement] fichiers: 98 | snapshots: 6 | lignes: 201001


## Étape 5 · Maintenance et export

In [21]:
spark.sql("""CALL demo.system.rewrite_data_files(
  table => 'silver.orders',
  options => map('target-file-size-bytes', '134217728'))""").show(truncate=False)
stats("Étape 5 : après compaction")

+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|95                        |4                     |6807883              |0                      |
+--------------------------+----------------------+---------------------+-----------------------+

[Étape 5 : après compaction] fichiers: 7 | snapshots: 7 | lignes: 201001


In [22]:
# Fichiers de la table apres compaction : la colonne content distingue donnees (0) et suppressions de position (1)
spark.sql("""SELECT spec_id, content, partition, record_count
             FROM demo.silver.orders.files ORDER BY spec_id DESC, partition""").show(truncate=False)

+-------+-------+------------------+------------+
|spec_id|content|partition         |record_count|
+-------+-------+------------------+------------+
|1      |0      |{NULL, 678}       |38384       |
|1      |0      |{NULL, 679}       |68797       |
|1      |0      |{NULL, 680}       |66631       |
|1      |0      |{NULL, 681}       |27188       |
|1      |0      |{NULL, 681}       |1           |
|0      |1      |{2026-08-07, NULL}|262         |
|0      |1      |{2026-08-18, NULL}|738         |
+-------+-------+------------------+------------+



In [23]:
maintenant = spark.sql("SELECT date_format(current_timestamp(), 'yyyy-MM-dd HH:mm:ss')").collect()[0][0]
spark.sql(f"""CALL demo.system.expire_snapshots(
  table => 'silver.orders',
  older_than => TIMESTAMP '{maintenant}',
  retain_last => 5)""").show(truncate=False)
stats("Étape 5 : après expire_snapshots")

26/10/18 17:20:56 WARN S3FileIO: Unclosed S3FileIO instance created by:
	org.apache.iceberg.aws.s3.S3FileIO.initialize(S3FileIO.java:371)
	org.apache.iceberg.CatalogUtil.loadFileIO(CatalogUtil.java:369)
	org.apache.iceberg.rest.RESTSessionCatalog.newFileIO(RESTSessionCatalog.java:1042)
	org.apache.iceberg.rest.RESTSessionCatalog.initialize(RESTSessionCatalog.java:323)
	org.apache.iceberg.rest.RESTCatalog.initialize(RESTCatalog.java:78)
	org.apache.iceberg.CatalogUtil.loadCatalog(CatalogUtil.java:274)
	org.apache.iceberg.CatalogUtil.buildIcebergCatalog(CatalogUtil.java:328)
	org.apache.iceberg.spark.SparkCatalog.buildIcebergCatalog(SparkCatalog.java:153)
	org.apache.iceberg.spark.SparkCatalog.initialize(SparkCatalog.java:752)
	org.apache.spark.sql.connector.catalog.Catalogs$.load(Catalogs.scala:65)
	org.apache.spark.sql.connector.catalog.CatalogManager.$anonfun$catalog$1(CatalogManager.scala:54)
	scala.collection.mutable.HashMap.getOrElseUpdate(HashMap.scala:86)
	org.apache.spark.sql.co

+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|deleted_data_files_count|deleted_position_delete_files_count|deleted_equality_delete_files_count|deleted_manifest_files_count|deleted_manifest_lists_count|deleted_statistics_files_count|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|2                       |93                                 |0                                  |3                           |2                           |0                             |
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+

[Étape 5 : après expire_snapshots] fichiers: 7 | snapshots:

In [24]:
try:
    hc = spark.sparkContext._jsc.hadoopConfiguration()
    hc.set("fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    hc.set("fs.s3a.endpoint", "http://minio:9000")
    hc.set("fs.s3a.access.key", "admin")
    hc.set("fs.s3a.secret.key", "password")
    hc.set("fs.s3a.path.style.access", "true")
    hc.set("fs.s3a.connection.ssl.enabled", "false")
    spark.sql("CALL demo.system.remove_orphan_files(table => 'silver.orders')").show(truncate=False)
except Exception as e:
    print("remove_orphan_files n'a pas pu s'exécuter dans cet environnement :")
    print(str(e).split("\n")[0][:250])
stats("Étape 5 : après remove_orphan_files")

remove_orphan_files n'a pas pu s'exécuter dans cet environnement :
An error occurred while calling o36.sql.
[Étape 5 : après remove_orphan_files] fichiers: 7 | snapshots: 5 | lignes: 201001


In [25]:
import os
os.makedirs("/home/iceberg/notebooks/data/silver", exist_ok=True)
df = spark.table("demo.silver.orders").toPandas()
df.to_parquet("/home/iceberg/notebooks/data/silver/orders.parquet", index=False)
print(len(df), "lignes exportées")

201001 lignes exportées


## Récapitulatif (à recopier dans TP2.md)

In [27]:
import pandas as pd
print(pd.DataFrame(results, columns=["étape", "fichiers", "snapshots", "lignes"]).to_string(index=False))
print(f"\nMERGE copy-on-write : {t_cow:.1f} s | merge-on-read : {t_mor:.1f} s")

                                        étape  fichiers  snapshots  lignes
              Étape 2 : après écriture Silver        91          1  200000
          Étape 3 : après MERGE copy-on-write        92          2  200500
          Étape 3 : après MERGE merge-on-read        97          3  201000
           Étape 3 : après re-MERGE identique        97          4  201000
            Étape 4 : après DELETE + rollback        97          5  201000
          Étape 4 : après évolution du schéma        97          5  201000
Étape 4 : après changement de partitionnement        98          6  201001
                   Étape 5 : après compaction         7          7  201001
             Étape 5 : après expire_snapshots         7          5  201001
          Étape 5 : après remove_orphan_files         7          5  201001

MERGE copy-on-write : 11.1 s | merge-on-read : 7.2 s
